# Lesson 12.1 - MLOps Foundations & ML Lifecycle (toy pipeline demo)

This notebook demonstrates a lightweight MLOps loop:

- reproducible training/evaluation,
- run tracking,
- model promotion decision,
- lifecycle status summary.


## Objectives

1. Build a toy but realistic ML run pipeline.
2. Track experiments with explicit metadata.
3. Simulate a model registry promotion gate.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

## Step 1: Create Reproducible Dataset Split

In [2]:
X, y = make_classification(
    n_samples=2200,
    n_features=18,
    n_informative=10,
    n_redundant=4,
    weights=[0.78, 0.22],
    random_state=SEED,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

X_train.shape, X_test.shape

((1650, 18), (550, 18))

## Step 2: Define Run Tracking Structure

In [3]:
@dataclass
class RunRecord:
    run_id: str
    model_name: str
    params: Dict[str, float]
    auc: float
    f1: float
    promoted: bool = False


run_store: List[RunRecord] = []

## Step 3: Train Baseline and Candidate Models

In [4]:
def evaluate_model(model, model_name: str, params: Dict[str, float], run_id: str) -> RunRecord:
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return RunRecord(
        run_id=run_id,
        model_name=model_name,
        params=params,
        auc=float(roc_auc_score(y_test, proba)),
        f1=float(f1_score(y_test, pred)),
    )


baseline = evaluate_model(
    LogisticRegression(max_iter=1500, random_state=SEED),
    model_name="logistic_regression",
    params={"max_iter": 1500},
    run_id="run_001",
)

candidate = evaluate_model(
    RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED),
    model_name="random_forest",
    params={"n_estimators": 300, "max_depth": 8},
    run_id="run_002",
)

run_store.extend([baseline, candidate])
pd.DataFrame([asdict(r) for r in run_store])

,run_id,model_name,params,auc,f1,promoted
0,run_001,logistic_regression,{'max_iter': 1500},0.903718,0.672727,False
1,run_002,random_forest,"{'n_estimators': 300, 'max_depth': 8}",0.957557,0.777251,False


## Step 4: Promotion Gate (Simple Rule)

In [5]:
PROMOTION_RULE = {
    "min_auc": 0.90,
    "min_f1": 0.72,
}


def should_promote(run: RunRecord, rule: Dict[str, float]) -> bool:
    return run.auc >= rule["min_auc"] and run.f1 >= rule["min_f1"]


for r in run_store:
    r.promoted = should_promote(r, PROMOTION_RULE)

registry_df = pd.DataFrame([asdict(r) for r in run_store]).sort_values(["promoted", "auc"], ascending=False)
registry_df

,run_id,model_name,params,auc,f1,promoted
1,run_002,random_forest,"{'n_estimators': 300, 'max_depth': 8}",0.957557,0.777251,True
0,run_001,logistic_regression,{'max_iter': 1500},0.903718,0.672727,False


## Step 5: Lifecycle Status Summary

In [6]:
lifecycle_summary = {
    "data_version": "v1.0.0",
    "training_code_version": "lesson-12-demo",
    "candidate_run": registry_df.iloc[0]["run_id"],
    "candidate_model": registry_df.iloc[0]["model_name"],
    "promotion_decision": bool(registry_df.iloc[0]["promoted"]),
    "next_step": "Deploy to staging" if bool(registry_df.iloc[0]["promoted"]) else "Tune model and rerun",
}

lifecycle_summary

{'data_version': 'v1.0.0',
 'training_code_version': 'lesson-12-demo',
 'candidate_run': 'run_002',
 'candidate_model': 'random_forest',
 'promotion_decision': True,
 'next_step': 'Deploy to staging'}

## Production Case Studies & Exceptions

### Case 1: Marketing churn model with manual workflows
- Teams retrained ad hoc, causing inconsistent model behavior.
- Introducing run tracking + promotion gate stabilized releases.

### Case 2: Regulated credit model
- Candidate model had higher AUC but failed explainability governance checks.
- Promotion blocked despite metric gains.

### Case 3 (Exception): Low-impact internal model
- Full automation was unnecessary; lightweight scheduled retraining was sufficient.

## Interview Questions & Answers

1. **Q: What is MLOps lifecycle in practice?**
   **A:** Data, training, validation, registry, deployment, monitoring, and retraining.

2. **Q: Why track runs explicitly?**
   **A:** To ensure reproducibility, auditability, and reliable model promotion decisions.

3. **Q: What is model promotion gate?**
   **A:** A rule set that decides whether a model can advance to the next environment.

4. **Q: Why can best AUC model be rejected?**
   **A:** Governance, latency, or fairness constraints may fail.

5. **Q: What is CT in MLOps?**
   **A:** Continuous training triggered by data/performance changes.

6. **Q: How do you avoid random improvements due to chance?**
   **A:** Use fixed splits, seeded runs, and repeated validations.

7. **Q: What metadata should be logged per run?**
   **A:** Code version, data version, hyperparameters, metrics, and artifacts.

8. **Q: When is a simple process preferable to full MLOps automation?**
   **A:** Low-risk, low-frequency internal decisions.

9. **Q: What is the role of model registry?**
   **A:** Version and govern promotion across stages.

10. **Q: How do you tie model metrics to business impact?**
   **A:** Map offline metrics to operational KPIs and monitor both.
